# FE5108 Part 4 - Leveraged single-stock ETF analysis

This notebook answers Parts **4A-4F** for the GraniteShares 2x Long NVDA Daily ETF (NVDL). It compares NVDL with NVIDIA stock (NVDA) using the supplied daily total returns from February through August 2024.

## How to use this notebook

From the repository root, run `uv sync`, then execute all cells from top to bottom. The source workbook is read without modification. Generated CSV/JSON/Markdown outputs are local files under `part4/outputs/` and are ignored by Git.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((path for path in candidates if (path / 'part4' / '__init__.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from the goldman2 repository.')
sys.path.insert(0, str(ROOT))
pd.set_option('display.max_columns', 12)
pd.set_option('display.width', 140)
plt.style.use('seaborn-v0_8-whitegrid')
ROOT

In [ ]:
from part4.analysis import (
    FACTSHEET_NET_EXPENSE_RATIO,
    build_part4_results,
    interpretation_markdown,
)

## Data validation

The source contains decimal daily total returns. The loader sorts dates ascending and verifies the exact 147-observation assignment sample, labels, missing values and return units.

In [ ]:
results = build_part4_results()
validation = pd.Series({
    'Observations': results.validation.rows,
    'Start': results.validation.start,
    'End': results.validation.end,
    'Missing values': results.validation.missing_values,
}, name='Value').to_frame()
display(validation)

## 4A - Risk comparison

All metrics are daily and unannualized. VaR and CVaR are shown as positive loss magnitudes.

In [ ]:
risk_display = results.risk_metrics.copy()
display(risk_display.style.format({
    'Daily standard deviation': '{:.2%}',
    '95% VaR': '{:.2%}',
    '95% CVaR': '{:.2%}',
    '5% return cutoff': '{:.2%}',
}))
display((risk_display.loc['NVDL', ['Daily standard deviation', '95% VaR', '95% CVaR']]
         / risk_display.loc['NVDA', ['Daily standard deviation', '95% VaR', '95% CVaR']])
        .rename('NVDL / NVDA risk multiple').to_frame().style.format('{:.2f}x'))

## 4B-4C - Daily leverage regression and performance drag

We estimate `NVDL = alpha + beta * NVDA + error` with an intercept and report 95% OLS confidence intervals.

In [ ]:
display(results.regression.coefficients.style.format('{:.6f}'))
regression_summary = pd.Series({
    'Observations': results.regression.observations,
    'R-squared': results.regression.r_squared,
    'Annualized intercept (daily alpha x 252)': results.regression.annualized_intercept,
    'Factsheet net expense ratio': FACTSHEET_NET_EXPENSE_RATIO,
}, name='Value').to_frame()
display(regression_summary.style.format({'Value': lambda x: f'{x:.2%}' if isinstance(x, float) else str(x)}))

In [ ]:
daily = results.daily_returns
alpha = results.regression.coefficients.loc['Intercept', 'Estimate']
beta = results.regression.coefficients.loc['NVDA beta', 'Estimate']
grid = np.linspace(daily['NVDA'].min(), daily['NVDA'].max(), 200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(daily['NVDA'], daily['NVDL'], alpha=0.65, s=28, label='Daily observations')
ax.plot(grid, alpha + beta * grid, linewidth=2, label='Estimated regression')
ax.plot(grid, 2 * grid, linestyle='--', linewidth=1.8, label='Exact 2x line')
ax.set(title='NVDL daily returns versus NVDA', xlabel='NVDA daily return', ylabel='NVDL daily return')
ax.xaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
ax.yaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
ax.legend()
plt.show()

## 4D - Returns beyond one day

Daily returns are compounded within each month. The full-period table separates twice NVDA's cumulative return from an ideal fee-free daily-reset 2x path, then attributes the difference to daily path compounding and observed NVDL tracking/performance drag.

In [ ]:
display(results.monthly_returns.style.format('{:.2%}'))
display(results.full_period_returns.style.format('{:.2%}'))
ax = results.monthly_returns[['NVDA', 'NVDL', 'Twice NVDA cumulative return']].plot.bar(
    figsize=(10, 5), color=['#7F8C8D', '#1F4E78', '#5B9BD5']
)
ax.set(title='Monthly total returns', xlabel='Month', ylabel='Total return')
ax.yaxis.set_major_formatter(lambda value, _: f'{value:.0%}')
ax.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=0)
plt.show()

## 4E - Two-day compounding example

In [ ]:
display(results.two_day_example.style.format('{:.2%}'))

## 4F - Investor suitability

In [ ]:
display(results.suitability)

## Report-ready interpretation

In [ ]:
display(Markdown(interpretation_markdown(results)))